In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Ecommerce_Data_Warehouse") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.3.0-amzn-1


In [4]:
BUCKET = "<your-bucket-name>"

HISTORICAL_PATH = f"s3://{BUCKET}/processed/ecommerce/"

print("Historical path:", HISTORICAL_PATH)

Historical path: s3://<your-bucket-name>/processed/ecommerce/


In [5]:
historical = {}

folders = [
    "categories",
    "category_sales",
    "customer_sales",
    "customers",
    "fact_sales",
    "monthly_sales",
    "order_details",
    "orders",
    "payments",
    "product_sales",
    "products",
    "shipments"
]

for folder in folders:
    path = f"{HISTORICAL_PATH}{folder}/"
    historical[folder] = spark.read.parquet(path)
    
    print(f"\n===== {folder} =====")
    historical[folder].printSchema()
    print("Count:", historical[folder].count())


===== categories =====
root
 |-- categoryid: integer (nullable = true)
 |-- categoryname: string (nullable = true)

Count: 20

===== category_sales =====
root
 |-- categoryid: integer (nullable = true)
 |-- total_sales: decimal(33,2) (nullable = true)
 |-- total_quantity: long (nullable = true)

Count: 20

===== customer_sales =====
root
 |-- customerid: long (nullable = true)
 |-- total_sales: decimal(33,2) (nullable = true)

Count: 9861

===== customers =====
root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registrationdate: timestamp (nullable = true)
 |-- full_name: string (nullable = true)
 |-- total_sales: decimal(38,2) (nullable = true)
 |-- customer_segment: string (nullable = true)

Count: 10000

===== fact_sales =====
root
 |-- orderid: long (nullable = true)
 |-- customerid: long 

In [7]:
RAW_PATH = f"s3://{BUCKET}/raw/ecommerce/"

print("Raw path:", RAW_PATH)

Raw path: s3://<your-bucket-name>/raw/ecommerce/


In [8]:
raw = {}

raw_folders = [
    "departments",
    "employees",
    "suppliers",
    "shippers"
]

for folder in raw_folders:
    path = f"{RAW_PATH}{folder}.csv"
    
    raw[folder] = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .csv(path)
    
    print(f"\n===== {folder} =====")
    raw[folder].printSchema()
    print("Count:", raw[folder].count())


===== departments =====
root
 |-- DepartmentID: integer (nullable = true)
 |-- DepartmentName: string (nullable = true)

Count: 10

===== employees =====
root
 |-- EmployeeID: integer (nullable = true)
 |-- ManagerID: integer (nullable = true)
 |-- DepartmentID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Salary: double (nullable = true)
 |-- HireDate: timestamp (nullable = true)

Count: 200

===== suppliers =====
root
 |-- SupplierID: integer (nullable = true)
 |-- SupplierName: string (nullable = true)
 |-- Country: string (nullable = true)

Count: 100

===== shippers =====
root
 |-- ShipperID: integer (nullable = true)
 |-- CompanyName: string (nullable = true)

Count: 10


In [9]:
API_ORDERS_PATH = f"s3://{BUCKET}/api_raw/orders/"

new_orders_raw = spark.read.json(API_ORDERS_PATH)

print("Number of records:", new_orders_raw.count())

new_orders_raw.printSchema()

Number of records: 220
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- PaymentD

In [10]:
new_orders_raw.show(3, truncate=False)


+-----------------------------------------------------------------------------------------------------+--------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------+---------------------------------------------+-----------------------------------------------+
|metadata                                                                                             |order                                 |order_details                                                                                                                                                         |payment                                                             |shipment                                     |summary                                        |
+-----------------------------------------------------

In [11]:
from pyspark.sql import functions as F

new_order_details = new_orders_raw.select(
    F.col("order.CustomerID").alias("CustomerID"),
    F.col("order.OrderDate").alias("OrderDate"),
    F.col("order.OrderID").alias("OrderID"),
    F.col("order.Status").alias("Status"),
    F.explode("order_details").alias("detail")
).select(
    "CustomerID",
    "OrderDate",
    "OrderID",
    "Status",
    F.col("detail.OrderDetailID").alias("OrderDetailID"),
    F.col("detail.ProductID").alias("ProductID"),
    F.col("detail.Quantity").alias("Quantity"),
    F.col("detail.UnitPrice").alias("UnitPrice"),
    F.col("detail.Discount").alias("Discount")
)

new_order_details.printSchema()
print("Count:", new_order_details.count())

new_order_details.show(5, truncate=False)

root
 |-- CustomerID: long (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- OrderID: long (nullable = true)
 |-- Status: string (nullable = true)
 |-- OrderDetailID: long (nullable = true)
 |-- ProductID: long (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- Discount: double (nullable = true)

Count: 550
+----------+----------+-------+---------+-------------+---------+--------+---------+--------+
|CustomerID|OrderDate |OrderID|Status   |OrderDetailID|ProductID|Quantity|UnitPrice|Discount|
+----------+----------+-------+---------+-------------+---------+--------+---------+--------+
|6435      |2026-09-22|2000074|Confirmed|5000200      |146      |4       |246.3    |10.0    |
|6435      |2026-09-22|2000074|Confirmed|5000201      |592      |4       |365.09   |0.0     |
|6435      |2026-09-22|2000074|Confirmed|5000202      |648      |1       |813.65   |10.0    |
|6435      |2026-09-22|2000074|Confirmed|5000203      |276    

In [12]:
new_order_details_clean = new_order_details.select(
    F.col("OrderDetailID").cast("long").alias("orderdetailid"),
    F.col("OrderID").cast("long").alias("orderid"),
    F.col("ProductID").cast("long").alias("productid"),
    F.col("Quantity").cast("int").alias("quantity"),
    F.col("UnitPrice").cast("decimal(12,2)").alias("unitprice"),
    F.col("Discount").cast("int").alias("discount"),
    (
        F.col("Quantity") * F.col("UnitPrice")
    ).cast("decimal(23,2)").alias("total_amount")
)

new_order_details_clean.printSchema()
new_order_details_clean.show(5, truncate=False)

root
 |-- orderdetailid: long (nullable = true)
 |-- orderid: long (nullable = true)
 |-- productid: long (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unitprice: decimal(12,2) (nullable = true)
 |-- discount: integer (nullable = true)
 |-- total_amount: decimal(23,2) (nullable = true)

+-------------+-------+---------+--------+---------+--------+------------+
|orderdetailid|orderid|productid|quantity|unitprice|discount|total_amount|
+-------------+-------+---------+--------+---------+--------+------------+
|5000200      |2000074|146      |4       |246.30   |10      |985.20      |
|5000201      |2000074|592      |4       |365.09   |0       |1460.36     |
|5000202      |2000074|648      |1       |813.65   |10      |813.65      |
|5000203      |2000074|276      |5       |858.16   |5       |4290.80     |
|5000300      |2000117|665      |5       |929.79   |15      |4648.95     |
+-------------+-------+---------+--------+---------+--------+------------+
only showing top 5 

In [14]:
print("=== HISTORICAL ===")
historical["order_details"].printSchema()

print("\n=== NEW ===")
new_order_details_clean.printSchema()

=== HISTORICAL ===
root
 |-- orderdetailid: long (nullable = true)
 |-- orderid: long (nullable = true)
 |-- productid: long (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unitprice: decimal(12,2) (nullable = true)
 |-- discount: integer (nullable = true)
 |-- total_amount: decimal(23,2) (nullable = true)


=== NEW ===
root
 |-- orderdetailid: long (nullable = true)
 |-- orderid: long (nullable = true)
 |-- productid: long (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unitprice: decimal(12,2) (nullable = true)
 |-- discount: integer (nullable = true)
 |-- total_amount: decimal(23,2) (nullable = true)


In [15]:
all_order_details = historical["order_details"].unionByName(
    new_order_details_clean
)

print("Historical count:", historical["order_details"].count())
print("New count:", new_order_details_clean.count())
print("Combined count:", all_order_details.count())

all_order_details.show(5, truncate=False)

Historical count: 100000
New count: 550
Combined count: 100550
+-------------+-------+---------+--------+---------+--------+------------+
|orderdetailid|orderid|productid|quantity|unitprice|discount|total_amount|
+-------------+-------+---------+--------+---------+--------+------------+
|1            |19982  |137      |4       |662.58   |0       |2650.32     |
|2            |4628   |793      |4       |1963.73  |15      |7854.92     |
|3            |8945   |947      |5       |644.63   |0       |3223.15     |
|4            |49362  |611      |5       |1373.52  |0       |6867.60     |
|5            |48941  |121      |6       |2573.48  |0       |15440.88    |
+-------------+-------+---------+--------+---------+--------+------------+
only showing top 5 rows


In [16]:
historical["orders"].printSchema()
historical["orders"].show(3, truncate=False)

root
 |-- orderid: long (nullable = true)
 |-- customerid: long (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- status: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- order_total: decimal(33,2) (nullable = true)
 |-- order_value_category: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+-------+----------+----------+---------+-------+---+-----------+-----------+--------------------+----+-----+
|orderid|customerid|orderdate |status   |quarter|day|day_of_week|order_total|order_value_category|year|month|
+-------+----------+----------+---------+-------+---+-----------+-----------+--------------------+----+-----+
|137    |8909      |2024-03-16|SHIPPED  |1      |16 |7          |2495.12    |HIGH                |2024|3    |
|341    |7701      |2024-03-25|PENDING  |1      |25 |2          |18754.75   |HIGH                |2024|

In [17]:
new_orders = new_orders_raw.select(
    F.col("order.OrderID").cast("long").alias("orderid"),
    F.col("order.CustomerID").cast("long").alias("customerid"),
    F.to_date("order.OrderDate").alias("orderdate"),
    F.upper(F.trim(F.col("order.Status"))).alias("status"),
    F.col("summary.total_amount").cast("decimal(33,2)").alias("order_total")
)

In [18]:
new_orders = new_orders \
    .withColumn("quarter", F.quarter("orderdate")) \
    .withColumn("day", F.dayofmonth("orderdate")) \
    .withColumn("day_of_week", F.dayofweek("orderdate")) \
    .withColumn("year", F.year("orderdate")) \
    .withColumn("month", F.month("orderdate")) \
    .withColumn(
        "order_value_category",
        F.when(F.col("order_total") >= 1000, "HIGH")
         .when(F.col("order_total") >= 500, "MEDIUM")
         .otherwise("LOW")
    )

In [19]:
new_orders = new_orders.select(
    "orderid",
    "customerid",
    "orderdate",
    "status",
    "quarter",
    "day",
    "day_of_week",
    "order_total",
    "order_value_category",
    "year",
    "month"
)

new_orders.printSchema()
new_orders.show(5, truncate=False)

root
 |-- orderid: long (nullable = true)
 |-- customerid: long (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- status: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- order_total: decimal(33,2) (nullable = true)
 |-- order_value_category: string (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

+-------+----------+----------+---------+-------+---+-----------+-----------+--------------------+----+-----+
|orderid|customerid|orderdate |status   |quarter|day|day_of_week|order_total|order_value_category|year|month|
+-------+----------+----------+---------+-------+---+-----------+-----------+--------------------+----+-----+
|2000074|6435      |2026-09-22|CONFIRMED|3      |22 |3          |8207.36    |HIGH                |2026|9    |
|2000117|7158      |2026-09-22|CONFIRMED|3      |22 |3          |9888.29    |HIGH                |2026

In [20]:
all_orders = historical["orders"].unionByName(
    new_orders
)

print("Historical count:", historical["orders"].count())
print("New count:", new_orders.count())
print("Combined count:", all_orders.count())

all_orders.show(5, truncate=False)

Historical count: 50000
New count: 220
Combined count: 50220
+-------+----------+----------+---------+-------+---+-----------+-----------+--------------------+----+-----+
|orderid|customerid|orderdate |status   |quarter|day|day_of_week|order_total|order_value_category|year|month|
+-------+----------+----------+---------+-------+---+-----------+-----------+--------------------+----+-----+
|137    |8909      |2024-03-16|SHIPPED  |1      |16 |7          |2495.12    |HIGH                |2024|3    |
|341    |7701      |2024-03-25|PENDING  |1      |25 |2          |18754.75   |HIGH                |2024|3    |
|366    |9744      |2024-03-23|CANCELLED|1      |23 |7          |31701.14   |HIGH                |2024|3    |
|398    |6042      |2024-03-02|PENDING  |1      |2  |7          |7264.28    |HIGH                |2024|3    |
|441    |6091      |2024-03-01|SHIPPED  |1      |1  |6          |9794.93    |HIGH                |2024|3    |
+-------+----------+----------+---------+-------+---+------

In [21]:
historical["payments"].printSchema()
historical["payments"].show(5, truncate=False)

root
 |-- paymentid: integer (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- paymentmethod: string (nullable = true)
 |-- paymentdate: timestamp (nullable = true)
 |-- amount: double (nullable = true)

+---------+-------+-------------+-------------------+-------+
|paymentid|orderid|paymentmethod|paymentdate        |amount |
+---------+-------+-------------+-------------------+-------+
|1        |43844  |Bank Transfer|2025-11-28 00:00:00|4967.26|
|2        |49938  |Cash         |2025-02-20 00:00:00|1551.2 |
|3        |8582   |Credit Card  |2024-10-16 00:00:00|2955.92|
|4        |32132  |Credit Card  |2024-07-31 00:00:00|3780.85|
|5        |28523  |PayPal       |2025-07-19 00:00:00|1736.92|
+---------+-------+-------------+-------------------+-------+
only showing top 5 rows


In [22]:
new_payments = new_orders_raw.select(
    F.col("payment.PaymentID").cast("int").alias("paymentid"),
    F.col("payment.OrderID").cast("int").alias("orderid"),
    F.trim(F.col("payment.PaymentMethod")).alias("paymentmethod"),
    F.to_timestamp("payment.PaymentDate").alias("paymentdate"),
    F.col("payment.Amount").cast("double").alias("amount")
)

new_payments.printSchema()
new_payments.show(5, truncate=False)

root
 |-- paymentid: integer (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- paymentmethod: string (nullable = true)
 |-- paymentdate: timestamp (nullable = true)
 |-- amount: double (nullable = true)

+---------+-------+----------------+-------------------+-------+
|paymentid|orderid|paymentmethod   |paymentdate        |amount |
+---------+-------+----------------+-------------------+-------+
|2000074  |2000074|Cash on Delivery|2026-09-22 00:00:00|8207.36|
|2000117  |2000117|Cash on Delivery|2026-09-22 00:00:00|9888.29|
|2000175  |2000175|Cash on Delivery|2026-09-22 00:00:00|6241.92|
|2000047  |2000047|Cash on Delivery|2026-09-22 00:00:00|4614.73|
|2000065  |2000065|Cash on Delivery|2026-09-22 00:00:00|5961.94|
+---------+-------+----------------+-------------------+-------+
only showing top 5 rows


In [23]:
id="8d2f1a"
existing_payment_orders = historical["payments"].select("orderid").distinct()

new_payments_only = new_payments.join(
    existing_payment_orders,
    on="orderid",
    how="left_anti"
)

print("New API payments:", new_payments.count())
print("New payments after duplicate check:", new_payments_only.count())

New API payments: 220
New payments after duplicate check: 220


In [24]:
all_payments = historical["payments"].unionByName(
    new_payments_only
)

print("Historical payments:", historical["payments"].count())
print("New payments:", new_payments_only.count())
print("Combined payments:", all_payments.count())

Historical payments: 45000
New payments: 220
Combined payments: 45220


In [25]:
historical["shipments"].printSchema()
historical["shipments"].show(5, truncate=False)

root
 |-- shipmentid: integer (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- shipperid: integer (nullable = true)
 |-- shipdate: timestamp (nullable = true)
 |-- deliverydate: timestamp (nullable = true)

+----------+-------+---------+-------------------+-------------------+
|shipmentid|orderid|shipperid|shipdate           |deliverydate       |
+----------+-------+---------+-------------------+-------------------+
|1         |26173  |6        |2026-08-11 00:00:00|2026-08-19 00:00:00|
|2         |37405  |3        |2026-04-22 00:00:00|2026-05-01 00:00:00|
|3         |37923  |8        |2024-12-01 00:00:00|2024-12-10 00:00:00|
|4         |36312  |7        |2026-08-22 00:00:00|2026-08-26 00:00:00|
|5         |15801  |10       |2026-03-06 00:00:00|2026-03-09 00:00:00|
+----------+-------+---------+-------------------+-------------------+
only showing top 5 rows


In [26]:
new_shipments = new_orders_raw.select(
    F.col("shipment.ShipmentID").cast("int").alias("shipmentid"),
    F.col("shipment.OrderID").cast("int").alias("orderid"),
    F.col("shipment.ShipperID").cast("int").alias("shipperid"),
    F.to_timestamp("shipment.ShipDate").alias("shipdate"),
    F.to_timestamp("shipment.DeliveryDate").alias("deliverydate")
).filter(
    F.col("shipmentid").isNotNull()
)

new_shipments.printSchema()
new_shipments.show(5, truncate=False)

root
 |-- shipmentid: integer (nullable = true)
 |-- orderid: integer (nullable = true)
 |-- shipperid: integer (nullable = true)
 |-- shipdate: timestamp (nullable = true)
 |-- deliverydate: timestamp (nullable = true)

+----------+-------+---------+-------------------+-------------------+
|shipmentid|orderid|shipperid|shipdate           |deliverydate       |
+----------+-------+---------+-------------------+-------------------+
|500065    |2000074|6        |2026-09-23 00:00:00|2026-09-28 00:00:00|
|500107    |2000117|10       |2026-09-23 00:00:00|2026-09-25 00:00:00|
|500164    |2000175|7        |2026-09-23 00:00:00|2026-09-28 00:00:00|
|500042    |2000047|1        |2026-09-23 00:00:00|2026-09-25 00:00:00|
|500059    |2000065|2        |2026-09-23 00:00:00|2026-09-27 00:00:00|
+----------+-------+---------+-------------------+-------------------+
only showing top 5 rows


In [27]:
existing_shipment_orders = historical["shipments"].select("orderid").distinct()

new_shipments_only = new_shipments.join(
    existing_shipment_orders,
    on="orderid",
    how="left_anti"
)

print("New API shipments:", new_shipments.count())
print("New shipments after duplicate check:", new_shipments_only.count())

New API shipments: 208
New shipments after duplicate check: 208


In [28]:
all_shipments = historical["shipments"].unionByName(
    new_shipments_only
)

print("Historical shipments:", historical["shipments"].count())
print("New shipments:", new_shipments_only.count())
print("Combined shipments:", all_shipments.count())

Historical shipments: 40000
New shipments: 208
Combined shipments: 40208


In [29]:
historical["customers"].printSchema()
historical["customers"].show(5, truncate=False)

root
 |-- customerid: integer (nullable = true)
 |-- firstname: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registrationdate: timestamp (nullable = true)
 |-- full_name: string (nullable = true)
 |-- total_sales: decimal(38,2) (nullable = true)
 |-- customer_segment: string (nullable = true)

+----------+---------+--------+----------------------------+---------+-------+-------------------+--------------+-----------+----------------+
|customerid|firstname|lastname|email                       |city     |country|registrationdate   |full_name     |total_sales|customer_segment|
+----------+---------+--------+----------------------------+---------+-------+-------------------+--------------+-----------+----------------+
|28        |Deborah  |Lynch   |deborah.lynch28@example.com |Cairo    |Jordan |2020-05-11 00:00:00|Deborah Lynch |131189.47  |VIP           

In [30]:
raw["customers"] = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{RAW_PATH}customers.csv")

raw["customers"].printSchema()
raw["customers"].show(5, truncate=False)

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: timestamp (nullable = true)

+----------+---------+--------+-----------------------------+------+--------------+-------------------+
|CustomerID|FirstName|LastName|Email                        |City  |Country       |RegistrationDate   |
+----------+---------+--------+-----------------------------+------+--------------+-------------------+
|1         |Danielle |Johnson |danielle.johnson1@example.com|Giza  |Egypt         |2023-01-31 00:00:00|
|2         |Joshua   |Walker  |joshua.walker2@example.com   |Dubai |Jordan        |2021-07-25 00:00:00|
|3         |Jill     |Rhodes  |jill.rhodes3@example.com     |Giza  |United Kingdom|2020-12-22 00:00:00|
|4         |Patricia |Miller  |patricia.miller4@example.com |London|Germany       |

In [31]:
from pyspark.sql.window import Window

customer_window = Window.orderBy("CustomerID")

dim_customer = raw["customers"].select(
    F.row_number().over(customer_window).cast("long").alias("customer_key"),
    F.col("CustomerID").cast("int").alias("customer_id"),
    F.trim(F.col("FirstName")).alias("first_name"),
    F.trim(F.col("LastName")).alias("last_name"),
    F.lower(F.trim(F.col("Email"))).alias("email"),
    F.lit(None).cast("string").alias("phone"),
    F.trim(F.col("City")).alias("city"),
    F.trim(F.col("Country")).alias("country"),
    F.col("RegistrationDate").cast("timestamp").alias("registration_date")
)

dim_customer.printSchema()
dim_customer.show(5, truncate=False)

root
 |-- customer_key: long (nullable = false)
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: timestamp (nullable = true)

+------------+-----------+----------+---------+-----------------------------+-----+------+--------------+-------------------+
|customer_key|customer_id|first_name|last_name|email                        |phone|city  |country       |registration_date  |
+------------+-----------+----------+---------+-----------------------------+-----+------+--------------+-------------------+
|1           |1          |Danielle  |Johnson  |danielle.johnson1@example.com|null |Giza  |Egypt         |2023-01-31 00:00:00|
|2           |2          |Joshua    |Walker   |joshua.walker2@example.com   |null |Dubai |Jordan        |2021-07-25 00:

In [32]:
raw["categories"] = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{RAW_PATH}categories.csv")

raw["categories"].printSchema()
raw["categories"].show(5, truncate=False)

root
 |-- CategoryID: integer (nullable = true)
 |-- CategoryName: string (nullable = true)

+----------+---------------+
|CategoryID|CategoryName   |
+----------+---------------+
|1         |Electronics    |
|2         |Computers      |
|3         |Mobile Phones  |
|4         |Home Appliances|
|5         |Furniture      |
+----------+---------------+
only showing top 5 rows


In [33]:
category_window = Window.orderBy("CategoryID")

dim_category = raw["categories"].select(
    F.row_number().over(category_window).cast("long").alias("category_key"),
    F.col("CategoryID").cast("int").alias("category_id"),
    F.trim(F.col("CategoryName")).alias("category_name")
)

dim_category.printSchema()
dim_category.show(5, truncate=False)

root
 |-- category_key: long (nullable = false)
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)

+------------+-----------+---------------+
|category_key|category_id|category_name  |
+------------+-----------+---------------+
|1           |1          |Electronics    |
|2           |2          |Computers      |
|3           |3          |Mobile Phones  |
|4           |4          |Home Appliances|
|5           |5          |Furniture      |
+------------+-----------+---------------+
only showing top 5 rows


In [34]:
department_window = Window.orderBy("DepartmentID")

dim_department = raw["departments"].select(
    F.row_number().over(department_window).cast("long").alias("department_key"),
    F.col("DepartmentID").cast("int").alias("department_id"),
    F.trim(F.col("DepartmentName")).alias("department_name")
)

dim_department.printSchema()
dim_department.show(5, truncate=False)

root
 |-- department_key: long (nullable = false)
 |-- department_id: integer (nullable = true)
 |-- department_name: string (nullable = true)

+--------------+-------------+---------------+
|department_key|department_id|department_name|
+--------------+-------------+---------------+
|1             |1            |IT             |
|2             |2            |Finance        |
|3             |3            |Sales          |
|4             |4            |Marketing      |
|5             |5            |HR             |
+--------------+-------------+---------------+
only showing top 5 rows


In [35]:
supplier_window = Window.orderBy("SupplierID")

dim_supplier = raw["suppliers"].select(
    F.row_number().over(supplier_window).cast("long").alias("supplier_key"),
    F.col("SupplierID").cast("int").alias("supplier_id"),
    F.trim(F.col("SupplierName")).alias("supplier_name"),
    F.trim(F.col("Country")).alias("country")
)

dim_supplier.printSchema()
dim_supplier.show(5, truncate=False)

root
 |-- supplier_key: long (nullable = false)
 |-- supplier_id: integer (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- country: string (nullable = true)

+------------+-----------+---------------------+-------+
|supplier_key|supplier_id|supplier_name        |country|
+------------+-----------+---------------------+-------+
|1           |1          |Digital World 1      |Jordan |
|2           |2          |Global Wholesale 2   |Germany|
|3           |3          |International Goods 3|Jordan |
|4           |4          |Smart Products Ltd 4 |UAE    |
|5           |5          |Prime Suppliers 5    |Germany|
+------------+-----------+---------------------+-------+
only showing top 5 rows


In [36]:
employee_window = Window.orderBy("EmployeeID")

dim_employee = raw["employees"].select(
    F.row_number().over(employee_window).cast("long").alias("employee_key"),
    F.col("EmployeeID").cast("int").alias("employee_id"),
    F.col("ManagerID").cast("int").alias("manager_id"),
    F.col("DepartmentID").cast("int").alias("department_id"),
    F.trim(F.col("FirstName")).alias("first_name"),
    F.trim(F.col("LastName")).alias("last_name"),
    F.col("Salary").cast("decimal(12,2)").alias("salary"),
    F.col("HireDate").cast("timestamp").alias("hire_date")
)

dim_employee.printSchema()
dim_employee.show(5, truncate=False)

root
 |-- employee_key: long (nullable = false)
 |-- employee_id: integer (nullable = true)
 |-- manager_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- salary: decimal(12,2) (nullable = true)
 |-- hire_date: timestamp (nullable = true)

+------------+-----------+----------+-------------+----------+---------+-------+-------------------+
|employee_key|employee_id|manager_id|department_id|first_name|last_name|salary |hire_date          |
+------------+-----------+----------+-------------+----------+---------+-------+-------------------+
|1           |1          |null      |5            |Nicole    |Porter   |8829.31|2021-03-16 00:00:00|
|2           |2          |1         |3            |Matthew   |Scott    |5994.40|2022-11-23 00:00:00|
|3           |3          |1         |5            |Samuel    |Jones    |6582.26|2018-04-01 00:00:00|
|4           |4          |3         |5  

In [37]:
shipper_window = Window.orderBy("ShipperID")

dim_shipper = raw["shippers"].select(
    F.row_number().over(shipper_window).cast("long").alias("shipper_key"),
    F.col("ShipperID").cast("int").alias("shipper_id"),
    F.trim(F.col("CompanyName")).alias("company_name")
)

dim_shipper.printSchema()
dim_shipper.show(5, truncate=False)

root
 |-- shipper_key: long (nullable = false)
 |-- shipper_id: integer (nullable = true)
 |-- company_name: string (nullable = true)

+-----------+----------+----------------+
|shipper_key|shipper_id|company_name    |
+-----------+----------+----------------+
|1          |1         |DHL             |
|2          |2         |FedEx           |
|3          |3         |Aramex          |
|4          |4         |UPS             |
|5          |5         |Amazon Logistics|
+-----------+----------+----------------+
only showing top 5 rows


In [39]:
payment_method_window = Window.orderBy("payment_method")

dim_payment_method = all_payments.select(
    F.trim(F.col("paymentmethod")).alias("payment_method")
).distinct().select(
    F.row_number().over(payment_method_window).cast("long").alias("payment_method_key"),
    F.col("payment_method")
)

dim_payment_method.printSchema()
dim_payment_method.show(truncate=False)

root
 |-- payment_method_key: long (nullable = false)
 |-- payment_method: string (nullable = true)

+------------------+----------------+
|payment_method_key|payment_method  |
+------------------+----------------+
|1                 |Bank Transfer   |
|2                 |Cash            |
|3                 |Cash on Delivery|
|4                 |Credit Card     |
|5                 |Debit Card      |
|6                 |PayPal          |
+------------------+----------------+


In [40]:
order_status_window = Window.orderBy("order_status")

dim_order_status = all_orders.select(
    F.trim(F.col("status")).alias("order_status")
).distinct().select(
    F.row_number().over(order_status_window).cast("long").alias("order_status_key"),
    F.col("order_status")
)

dim_order_status.printSchema()
dim_order_status.show(truncate=False)

root
 |-- order_status_key: long (nullable = false)
 |-- order_status: string (nullable = true)

+----------------+------------+
|order_status_key|order_status|
+----------------+------------+
|1               |CANCELLED   |
|2               |CONFIRMED   |
|3               |DELIVERED   |
|4               |PENDING     |
|5               |SHIPPED     |
+----------------+------------+


In [41]:
raw["products"] = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(f"{RAW_PATH}products.csv")

raw["products"].printSchema()
raw["products"].show(5, truncate=False)

root
 |-- ProductID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Stock: integer (nullable = true)

+---------+----------+-----------------+------+-------+-------+-----+
|ProductID|CategoryID|ProductName      |Brand |Price  |Cost   |Stock|
+---------+----------+-----------------+------+-------+-------+-----+
|1        |12        |Bose Printer 1   |Bose  |261.16 |140.65 |399  |
|2        |3         |Lenovo Mouse 2   |Lenovo|1874.2 |1460.92|155  |
|3        |12        |Sony Headphones 3|Sony  |1272.54|846.77 |315  |
|4        |11        |LG Smart Watch 4 |LG    |1180.67|637.81 |95   |
|5        |16        |Bose Television 5|Bose  |253.18 |165.51 |375  |
+---------+----------+-----------------+------+-------+-------+-----+
only showing top 5 rows


In [42]:
historical["products"].printSchema()
historical["products"].show(5, truncate=False)

root
 |-- productid: integer (nullable = true)
 |-- categoryid: integer (nullable = true)
 |-- productname: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- stock: integer (nullable = true)

+---------+----------+-----------------+------+-------+-------+-----+
|productid|categoryid|productname      |brand |price  |cost   |stock|
+---------+----------+-----------------+------+-------+-------+-----+
|1        |12        |Bose Printer 1   |Bose  |261.16 |140.65 |399  |
|2        |3         |Lenovo Mouse 2   |Lenovo|1874.2 |1460.92|155  |
|3        |12        |Sony Headphones 3|Sony  |1272.54|846.77 |315  |
|4        |11        |LG Smart Watch 4 |LG    |1180.67|637.81 |95   |
|5        |16        |Bose Television 5|Bose  |253.18 |165.51 |375  |
+---------+----------+-----------------+------+-------+-------+-----+
only showing top 5 rows


In [43]:
product_window = Window.orderBy("ProductID")

dim_product = raw["products"].alias("p") \
    .join(
        dim_category.alias("c"),
        F.col("p.CategoryID") == F.col("c.category_id"),
        "left"
    ) \
    .select(
        F.row_number().over(product_window).cast("long").alias("product_key"),
        F.col("p.ProductID").cast("int").alias("product_id"),
        F.col("c.category_key").cast("long").alias("category_key"),
        F.trim(F.col("p.ProductName")).alias("product_name"),
        F.trim(F.col("p.Brand")).alias("brand"),
        F.col("p.Price").cast("double").alias("price"),
        F.col("p.Cost").cast("double").alias("cost"),
        F.col("p.Stock").cast("int").alias("stock")
    )

dim_product.printSchema()
dim_product.show(5, truncate=False)

root
 |-- product_key: long (nullable = false)
 |-- product_id: integer (nullable = true)
 |-- category_key: long (nullable = true)
 |-- product_name: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- cost: double (nullable = true)
 |-- stock: integer (nullable = true)

+-----------+----------+------------+-----------------+------+-------+-------+-----+
|product_key|product_id|category_key|product_name     |brand |price  |cost   |stock|
+-----------+----------+------------+-----------------+------+-------+-------+-----+
|1          |1         |12          |Bose Printer 1   |Bose  |261.16 |140.65 |399  |
|2          |2         |3           |Lenovo Mouse 2   |Lenovo|1874.2 |1460.92|155  |
|3          |3         |12          |Sony Headphones 3|Sony  |1272.54|846.77 |315  |
|4          |4         |11          |LG Smart Watch 4 |LG    |1180.67|637.81 |95   |
|5          |5         |16          |Bose Television 5|Bose  |253.18 |165.51 |37

In [44]:
date_range = all_orders.select(
    F.min("orderdate").alias("min_date"),
    F.max("orderdate").alias("max_date")
).collect()[0]

print("Min date:", date_range["min_date"])
print("Max date:", date_range["max_date"])

Min date: 2024-01-01
Max date: 2026-09-22


In [45]:
date_df = spark.sql("""
    SELECT explode(
        sequence(
            to_date('2024-01-01'),
            to_date('2026-09-22'),
            interval 1 day
        )
    ) AS full_date
""")

dim_date = date_df.select(
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("full_date"),
    F.dayofmonth("full_date").alias("day"),
    F.month("full_date").alias("month"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.quarter("full_date").alias("quarter"),
    F.year("full_date").alias("year"),
    F.weekofyear("full_date").alias("week"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.dayofweek("full_date").isin([1, 7]).alias("is_weekend")
)

dim_date.printSchema()
dim_date.show(5, truncate=False)

root
 |-- date_key: integer (nullable = true)
 |-- full_date: date (nullable = false)
 |-- day: integer (nullable = false)
 |-- month: integer (nullable = false)
 |-- month_name: string (nullable = false)
 |-- quarter: integer (nullable = false)
 |-- year: integer (nullable = false)
 |-- week: integer (nullable = false)
 |-- day_name: string (nullable = false)
 |-- is_weekend: boolean (nullable = false)

+--------+----------+---+-----+----------+-------+----+----+---------+----------+
|date_key|full_date |day|month|month_name|quarter|year|week|day_name |is_weekend|
+--------+----------+---+-----+----------+-------+----+----+---------+----------+
|20240101|2024-01-01|1  |1    |January   |1      |2024|1   |Monday   |false     |
|20240102|2024-01-02|2  |1    |January   |1      |2024|1   |Tuesday  |false     |
|20240103|2024-01-03|3  |1    |January   |1      |2024|1   |Wednesday|false     |
|20240104|2024-01-04|4  |1    |January   |1      |2024|1   |Thursday |false     |
|20240105|2024-01-

In [46]:
fact_order = all_orders.alias("o") \
    .join(
        dim_customer.alias("c"),
        F.col("o.customerid") == F.col("c.customer_id"),
        "left"
    ) \
    .join(
        dim_date.alias("d"),
        F.col("o.orderdate") == F.col("d.full_date"),
        "left"
    ) \
    .join(
        dim_order_status.alias("s"),
        F.col("o.status") == F.col("s.order_status"),
        "left"
    ) \
    .select(
        F.col("o.orderid").alias("order_id"),
        F.col("c.customer_key"),
        F.col("d.date_key"),
        F.col("s.order_status_key"),
        F.col("o.order_total"),
        F.col("o.order_value_category")
    )

fact_order.printSchema()
fact_order.show(5, truncate=False)

root
 |-- order_id: long (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_status_key: long (nullable = true)
 |-- order_total: decimal(33,2) (nullable = true)
 |-- order_value_category: string (nullable = true)

+--------+------------+--------+----------------+-----------+--------------------+
|order_id|customer_key|date_key|order_status_key|order_total|order_value_category|
+--------+------------+--------+----------------+-----------+--------------------+
|137     |8909        |20240316|5               |2495.12    |HIGH                |
|341     |7701        |20240325|4               |18754.75   |HIGH                |
|366     |9744        |20240323|1               |31701.14   |HIGH                |
|398     |6042        |20240302|4               |7264.28    |HIGH                |
|441     |6091        |20240301|5               |9794.93    |HIGH                |
+--------+------------+--------+----------------+-----------+

In [47]:
fact_order_detail = all_order_details.alias("od") \
    .join(
        all_orders.alias("o"),
        F.col("od.orderid") == F.col("o.orderid"),
        "left"
    ) \
    .join(
        dim_product.alias("p"),
        F.col("od.productid") == F.col("p.product_id"),
        "left"
    ) \
    .join(
        dim_customer.alias("c"),
        F.col("o.customerid") == F.col("c.customer_id"),
        "left"
    ) \
    .join(
        dim_date.alias("d"),
        F.col("o.orderdate") == F.col("d.full_date"),
        "left"
    ) \
    .select(
        F.col("od.orderid").alias("order_id"),
        F.col("p.product_key"),
        F.col("c.customer_key"),
        F.col("d.date_key"),
        F.col("od.quantity"),
        F.col("od.unitprice").alias("unit_price"),
        F.col("od.discount"),
        (
            F.col("od.quantity") * F.col("od.unitprice")
        ).cast("decimal(23,2)").alias("gross_sales"),
        (
            F.col("od.quantity") *
            F.col("od.unitprice") *
            F.col("od.discount") / F.lit(100)
        ).cast("decimal(23,2)").alias("discount_amount"),
        (
            F.col("od.quantity") *
            F.col("od.unitprice") *
            (F.lit(1) - F.col("od.discount") / F.lit(100))
        ).cast("decimal(23,2)").alias("net_sales")
    )

fact_order_detail.printSchema()
fact_order_detail.show(5, truncate=False)

root
 |-- order_id: long (nullable = true)
 |-- product_key: long (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: decimal(12,2) (nullable = true)
 |-- discount: integer (nullable = true)
 |-- gross_sales: decimal(23,2) (nullable = true)
 |-- discount_amount: decimal(23,2) (nullable = true)
 |-- net_sales: decimal(23,2) (nullable = true)

+--------+-----------+------------+--------+--------+----------+--------+-----------+---------------+---------+
|order_id|product_key|customer_key|date_key|quantity|unit_price|discount|gross_sales|discount_amount|net_sales|
+--------+-----------+------------+--------+--------+----------+--------+-----------+---------------+---------+
|19982   |137        |2332        |20251104|4       |662.58    |0       |2650.32    |0.00           |2650.32  |
|4628    |793        |230         |20260723|4       |1963.73   |15      |7854.92    |1178.24        |

In [48]:
fact_payment = all_payments.alias("pay") \
    .join(
        all_orders.alias("o"),
        F.col("pay.orderid") == F.col("o.orderid"),
        "left"
    ) \
    .join(
        dim_customer.alias("c"),
        F.col("o.customerid") == F.col("c.customer_id"),
        "left"
    ) \
    .join(
        dim_date.alias("d"),
        F.to_date(F.col("pay.paymentdate")) == F.col("d.full_date"),
        "left"
    ) \
    .join(
        dim_payment_method.alias("pm"),
        F.col("pay.paymentmethod") == F.col("pm.payment_method"),
        "left"
    ) \
    .select(
        F.col("pay.paymentid").alias("payment_id"),
        F.col("pay.orderid").alias("order_id"),
        F.col("c.customer_key"),
        F.col("d.date_key"),
        F.col("pm.payment_method_key"),
        F.col("pay.paymentdate").alias("payment_date"),
        F.col("pay.amount").alias("payment_amount")
    )

In [49]:
fact_payment.printSchema()
fact_payment.show(5, truncate=False)

root
 |-- payment_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- payment_method_key: long (nullable = true)
 |-- payment_date: timestamp (nullable = true)
 |-- payment_amount: double (nullable = true)

+----------+--------+------------+--------+------------------+-------------------+--------------+
|payment_id|order_id|customer_key|date_key|payment_method_key|payment_date       |payment_amount|
+----------+--------+------------+--------+------------------+-------------------+--------------+
|1         |43844   |5970        |20251128|1                 |2025-11-28 00:00:00|4967.26       |
|2         |49938   |6582        |20250220|2                 |2025-02-20 00:00:00|1551.2        |
|3         |8582    |319         |20241016|4                 |2024-10-16 00:00:00|2955.92       |
|4         |32132   |3019        |20240731|4                 |2024-07-31 00:00:00|3780.85       |


In [50]:
fact_shipment = all_shipments.alias("sh") \
    .join(
        all_orders.alias("o"),
        F.col("sh.orderid") == F.col("o.orderid"),
        "left"
    ) \
    .join(
        dim_customer.alias("c"),
        F.col("o.customerid") == F.col("c.customer_id"),
        "left"
    ) \
    .join(
        dim_shipper.alias("sp"),
        F.col("sh.shipperid") == F.col("sp.shipper_id"),
        "left"
    ) \
    .join(
        dim_date.alias("d"),
        F.to_date(F.col("sh.shipdate")) == F.col("d.full_date"),
        "left"
    ) \
    .select(
        F.col("sh.shipmentid").alias("shipment_id"),
        F.col("sh.orderid").alias("order_id"),
        F.col("c.customer_key"),
        F.col("sp.shipper_key"),
        F.col("d.date_key"),
        F.col("sh.shipdate").alias("ship_date"),
        F.col("sh.deliverydate").alias("delivery_date"),
        F.datediff(
            F.to_date(F.col("sh.deliverydate")),
            F.to_date(F.col("sh.shipdate"))
        ).alias("delivery_duration_days")
    )

In [51]:
fact_shipment.printSchema()
fact_shipment.show(5, truncate=False)

root
 |-- shipment_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- customer_key: long (nullable = true)
 |-- shipper_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- ship_date: timestamp (nullable = true)
 |-- delivery_date: timestamp (nullable = true)
 |-- delivery_duration_days: integer (nullable = true)

+-----------+--------+------------+-----------+--------+-------------------+-------------------+----------------------+
|shipment_id|order_id|customer_key|shipper_key|date_key|ship_date          |delivery_date      |delivery_duration_days|
+-----------+--------+------------+-----------+--------+-------------------+-------------------+----------------------+
|1          |26173   |1092        |6          |20260811|2026-08-11 00:00:00|2026-08-19 00:00:00|8                     |
|2          |37405   |2733        |3          |20260422|2026-04-22 00:00:00|2026-05-01 00:00:00|9                     |
|3          |37923   |932         |8  

In [52]:
customer_sales_base = fact_order_detail.alias("fd") \
    .join(
        dim_product.alias("p"),
        F.col("fd.product_key") == F.col("p.product_key"),
        "left"
    )

fact_customer_sales = customer_sales_base \
    .groupBy(
        F.col("fd.customer_key"),
        F.col("fd.date_key")
    ) \
    .agg(
        F.countDistinct("fd.order_id").alias("order_count"),
        F.sum("fd.quantity").alias("quantity"),
        F.sum("fd.net_sales").cast("decimal(23,2)").alias("sales"),
        F.sum(
            F.col("fd.net_sales") -
            (F.col("fd.quantity") * F.col("p.cost"))
        ).cast("decimal(23,2)").alias("profit")
    )

In [53]:
fact_customer_sales.printSchema()
fact_customer_sales.show(5, truncate=False)

root
 |-- customer_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_count: long (nullable = false)
 |-- quantity: long (nullable = true)
 |-- sales: decimal(23,2) (nullable = true)
 |-- profit: decimal(23,2) (nullable = true)

+------------+--------+-----------+--------+--------+-------+
|customer_key|date_key|order_count|quantity|sales   |profit |
+------------+--------+-----------+--------+--------+-------+
|7653        |20260422|2          |18      |28792.97|9150.55|
|500         |20260407|1          |2       |2854.98 |1085.96|
|3831        |20260210|1          |9       |9196.37 |2734.19|
|6819        |20260401|1          |15      |8311.07 |3409.47|
|2396        |20250419|2          |10      |10423.14|1509.46|
+------------+--------+-----------+--------+--------+-------+
only showing top 5 rows


In [54]:
product_sales_base = fact_order_detail.alias("fd") \
    .join(
        dim_product.alias("p"),
        F.col("fd.product_key") == F.col("p.product_key"),
        "left"
    )

fact_product_sales = product_sales_base \
    .groupBy(
        F.col("fd.product_key"),
        F.col("fd.date_key")
    ) \
    .agg(
        F.countDistinct("fd.order_id").alias("order_count"),
        F.sum("fd.quantity").alias("quantity_sold"),
        F.sum("fd.gross_sales").cast("decimal(23,2)").alias("gross_sales"),
        F.sum("fd.discount_amount").cast("decimal(23,2)").alias("discount"),
        F.sum("fd.net_sales").cast("decimal(23,2)").alias("net_sales"),
        F.sum(
            F.col("fd.quantity") * F.col("p.cost")
        ).cast("decimal(23,2)").alias("cost"),
        F.sum(
            F.col("fd.net_sales") -
            (F.col("fd.quantity") * F.col("p.cost"))
        ).cast("decimal(23,2)").alias("profit")
    )

In [55]:
fact_product_sales.printSchema()
fact_product_sales.show(5, truncate=False)

root
 |-- product_key: long (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- order_count: long (nullable = false)
 |-- quantity_sold: long (nullable = true)
 |-- gross_sales: decimal(23,2) (nullable = true)
 |-- discount: decimal(23,2) (nullable = true)
 |-- net_sales: decimal(23,2) (nullable = true)
 |-- cost: decimal(23,2) (nullable = true)
 |-- profit: decimal(23,2) (nullable = true)

+-----------+--------+-----------+-------------+-----------+--------+---------+-------+-------+
|product_key|date_key|order_count|quantity_sold|gross_sales|discount|net_sales|cost   |profit |
+-----------+--------+-----------+-------------+-----------+--------+---------+-------+-------+
|696        |20240529|1          |5            |12700.35   |0.00    |12700.35 |7085.45|5614.90|
|359        |20260624|1          |8            |9268.00    |926.80  |8341.20  |6094.16|2247.04|
|623        |20240314|1          |8            |1809.92    |90.50   |1719.42  |1030.32|689.10 |
|814        |20251

In [56]:
dimensions = {
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_category": dim_category,
    "dim_department": dim_department,
    "dim_supplier": dim_supplier,
    "dim_employee": dim_employee,
    "dim_shipper": dim_shipper,
    "dim_date": dim_date,
    "dim_payment_method": dim_payment_method,
    "dim_order_status": dim_order_status
}

facts = {
    "fact_order": fact_order,
    "fact_order_detail": fact_order_detail,
    "fact_payment": fact_payment,
    "fact_shipment": fact_shipment,
    "fact_customer_sales": fact_customer_sales,
    "fact_product_sales": fact_product_sales
}

In [57]:
print("=== DIMENSIONS ===")

for name, df in dimensions.items():
    print(f"{name}: {df.count():,} rows")

print("\n=== FACTS ===")

for name, df in facts.items():
    print(f"{name}: {df.count():,} rows")

=== DIMENSIONS ===
dim_customer: 10,000 rows
dim_product: 1,000 rows
dim_category: 20 rows
dim_department: 10 rows
dim_supplier: 100 rows
dim_employee: 200 rows
dim_shipper: 10 rows
dim_date: 996 rows
dim_payment_method: 6 rows
dim_order_status: 5 rows

=== FACTS ===
fact_order: 50,220 rows
fact_order_detail: 100,550 rows
fact_payment: 45,220 rows
fact_shipment: 40,208 rows
fact_customer_sales: 43,276 rows
fact_product_sales: 95,513 rows


In [58]:
print("=== DIMENSION KEY CHECKS ===")

dimension_keys = {
    "dim_customer": "customer_id",
    "dim_product": "product_id",
    "dim_category": "category_id",
    "dim_department": "department_id",
    "dim_supplier": "supplier_id",
    "dim_employee": "employee_id",
    "dim_shipper": "shipper_id",
    "dim_date": "date_key",
    "dim_payment_method": "payment_method",
    "dim_order_status": "order_status"
}

for name, key in dimension_keys.items():
    df = dimensions[name]
    
    total = df.count()
    distinct = df.select(key).distinct().count()
    duplicates = total - distinct
    
    print(f"{name}: {duplicates} duplicate {key} values")

=== DIMENSION KEY CHECKS ===
dim_customer: 0 duplicate customer_id values
dim_product: 0 duplicate product_id values
dim_category: 0 duplicate category_id values
dim_department: 0 duplicate department_id values
dim_supplier: 0 duplicate supplier_id values
dim_employee: 0 duplicate employee_id values
dim_shipper: 0 duplicate shipper_id values
dim_date: 0 duplicate date_key values
dim_payment_method: 0 duplicate payment_method values
dim_order_status: 0 duplicate order_status values


In [60]:
print("\n=== FACT GRAIN CHECKS ===")

fact_grains = {
    "fact_order": ["order_id"],
    "fact_order_detail": ["order_id", "product_key"],
    "fact_payment": ["payment_id"],
    "fact_shipment": ["shipment_id"],
    "fact_customer_sales": ["customer_key", "date_key"],
    "fact_product_sales": ["product_key", "date_key"]
}

for name, keys in fact_grains.items():
    df = facts[name]
    
    total = df.count()
    distinct = df.select(*keys).distinct().count()
    duplicates = total - distinct
    
    print(f"{name}: {duplicates} duplicate grain combinations")


=== FACT GRAIN CHECKS ===
fact_order: 0 duplicate grain combinations
fact_order_detail: 95 duplicate grain combinations
fact_payment: 0 duplicate grain combinations
fact_shipment: 0 duplicate grain combinations
fact_customer_sales: 0 duplicate grain combinations
fact_product_sales: 0 duplicate grain combinations


In [61]:
total = all_order_details.count()

distinct_ids = all_order_details.select("orderdetailid").distinct().count()

print("Total order details:", total)
print("Distinct orderdetailid:", distinct_ids)
print("Duplicate orderdetailid:", total - distinct_ids)

Total order details: 100550
Distinct orderdetailid: 100550
Duplicate orderdetailid: 0


In [62]:
print("=== NULL FOREIGN KEY CHECKS ===")

checks = {
    "fact_order.customer_key": fact_order.filter(F.col("customer_key").isNull()).count(),
    "fact_order.date_key": fact_order.filter(F.col("date_key").isNull()).count(),
    "fact_order.order_status_key": fact_order.filter(F.col("order_status_key").isNull()).count(),

    "fact_order_detail.product_key": fact_order_detail.filter(F.col("product_key").isNull()).count(),
    "fact_order_detail.customer_key": fact_order_detail.filter(F.col("customer_key").isNull()).count(),
    "fact_order_detail.date_key": fact_order_detail.filter(F.col("date_key").isNull()).count(),

    "fact_payment.customer_key": fact_payment.filter(F.col("customer_key").isNull()).count(),
    "fact_payment.date_key": fact_payment.filter(F.col("date_key").isNull()).count(),
    "fact_payment.payment_method_key": fact_payment.filter(F.col("payment_method_key").isNull()).count(),

    "fact_shipment.customer_key": fact_shipment.filter(F.col("customer_key").isNull()).count(),
    "fact_shipment.shipper_key": fact_shipment.filter(F.col("shipper_key").isNull()).count(),
    "fact_shipment.date_key": fact_shipment.filter(F.col("date_key").isNull()).count(),

    "fact_customer_sales.customer_key": fact_customer_sales.filter(F.col("customer_key").isNull()).count(),
    "fact_customer_sales.date_key": fact_customer_sales.filter(F.col("date_key").isNull()).count(),

    "fact_product_sales.product_key": fact_product_sales.filter(F.col("product_key").isNull()).count(),
    "fact_product_sales.date_key": fact_product_sales.filter(F.col("date_key").isNull()).count()
}

for check, count in checks.items():
    print(f"{check}: {count}")

=== NULL FOREIGN KEY CHECKS ===
fact_order.customer_key: 0
fact_order.date_key: 0
fact_order.order_status_key: 0
fact_order_detail.product_key: 0
fact_order_detail.customer_key: 0
fact_order_detail.date_key: 0
fact_payment.customer_key: 0
fact_payment.date_key: 0
fact_payment.payment_method_key: 0
fact_shipment.customer_key: 0
fact_shipment.shipper_key: 0
fact_shipment.date_key: 208
fact_customer_sales.customer_key: 0
fact_customer_sales.date_key: 0
fact_product_sales.product_key: 0
fact_product_sales.date_key: 0


In [63]:
date_df = spark.sql("""
    SELECT explode(
        sequence(
            to_date('2024-01-01'),
            to_date('2026-09-28'),
            interval 1 day
        )
    ) AS full_date
""")

dim_date = date_df.select(
    F.date_format("full_date", "yyyyMMdd").cast("int").alias("date_key"),
    F.col("full_date"),
    F.dayofmonth("full_date").alias("day"),
    F.month("full_date").alias("month"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.quarter("full_date").alias("quarter"),
    F.year("full_date").alias("year"),
    F.weekofyear("full_date").alias("week"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.dayofweek("full_date").isin([1, 7]).alias("is_weekend")
)

In [64]:
fact_shipment.filter(
    F.col("date_key").isNull()
).count()

208


In [65]:
fact_shipment = all_shipments.alias("sh") \
    .join(
        all_orders.alias("o"),
        F.col("sh.orderid") == F.col("o.orderid"),
        "left"
    ) \
    .join(
        dim_customer.alias("c"),
        F.col("o.customerid") == F.col("c.customer_id"),
        "left"
    ) \
    .join(
        dim_shipper.alias("sp"),
        F.col("sh.shipperid") == F.col("sp.shipper_id"),
        "left"
    ) \
    .join(
        dim_date.alias("d"),
        F.to_date(F.col("sh.shipdate")) == F.col("d.full_date"),
        "left"
    ) \
    .select(
        F.col("sh.shipmentid").alias("shipment_id"),
        F.col("sh.orderid").alias("order_id"),
        F.col("c.customer_key"),
        F.col("sp.shipper_key"),
        F.col("d.date_key"),
        F.col("sh.shipdate").alias("ship_date"),
        F.col("sh.deliverydate").alias("delivery_date"),
        F.datediff(
            F.to_date(F.col("sh.deliverydate")),
            F.to_date(F.col("sh.shipdate"))
        ).alias("delivery_duration_days")
    )

In [66]:
fact_shipment.filter(
    F.col("date_key").isNull()
).count()

0


In [67]:
print("=== FOREIGN KEY INTEGRITY CHECK ===")

print(
    "fact_order → dim_customer:",
    fact_order.join(
        dim_customer,
        fact_order.customer_key == dim_customer.customer_key,
        "left_anti"
    ).count()
)

print(
    "fact_order → dim_date:",
    fact_order.join(
        dim_date,
        fact_order.date_key == dim_date.date_key,
        "left_anti"
    ).count()
)

print(
    "fact_order → dim_order_status:",
    fact_order.join(
        dim_order_status,
        fact_order.order_status_key == dim_order_status.order_status_key,
        "left_anti"
    ).count()
)

print(
    "fact_order_detail → dim_product:",
    fact_order_detail.join(
        dim_product,
        fact_order_detail.product_key == dim_product.product_key,
        "left_anti"
    ).count()
)

print(
    "fact_order_detail → dim_customer:",
    fact_order_detail.join(
        dim_customer,
        fact_order_detail.customer_key == dim_customer.customer_key,
        "left_anti"
    ).count()
)

print(
    "fact_order_detail → dim_date:",
    fact_order_detail.join(
        dim_date,
        fact_order_detail.date_key == dim_date.date_key,
        "left_anti"
    ).count()
)

print(
    "fact_payment → dim_customer:",
    fact_payment.join(
        dim_customer,
        fact_payment.customer_key == dim_customer.customer_key,
        "left_anti"
    ).count()
)

print(
    "fact_payment → dim_date:",
    fact_payment.join(
        dim_date,
        fact_payment.date_key == dim_date.date_key,
        "left_anti"
    ).count()
)

print(
    "fact_payment → dim_payment_method:",
    fact_payment.join(
        dim_payment_method,
        fact_payment.payment_method_key == dim_payment_method.payment_method_key,
        "left_anti"
    ).count()
)

print(
    "fact_shipment → dim_customer:",
    fact_shipment.join(
        dim_customer,
        fact_shipment.customer_key == dim_customer.customer_key,
        "left_anti"
    ).count()
)

print(
    "fact_shipment → dim_shipper:",
    fact_shipment.join(
        dim_shipper,
        fact_shipment.shipper_key == dim_shipper.shipper_key,
        "left_anti"
    ).count()
)

print(
    "fact_shipment → dim_date:",
    fact_shipment.join(
        dim_date,
        fact_shipment.date_key == dim_date.date_key,
        "left_anti"
    ).count()
)

=== FOREIGN KEY INTEGRITY CHECK ===
fact_order → dim_customer: 0
fact_order → dim_date: 0
fact_order → dim_order_status: 0
fact_order_detail → dim_product: 0
fact_order_detail → dim_customer: 0
fact_order_detail → dim_date: 0
fact_payment → dim_customer: 0
fact_payment → dim_date: 0
fact_payment → dim_payment_method: 0
fact_shipment → dim_customer: 0
fact_shipment → dim_shipper: 0
fact_shipment → dim_date: 0


In [69]:
dimensions = {
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_category": dim_category,
    "dim_department": dim_department,
    "dim_supplier": dim_supplier,
    "dim_employee": dim_employee,
    "dim_shipper": dim_shipper,
    "dim_date": dim_date,
    "dim_payment_method": dim_payment_method,
    "dim_order_status": dim_order_status
}

facts = {
    "fact_order": fact_order,
    "fact_order_detail": fact_order_detail,
    "fact_payment": fact_payment,
    "fact_shipment": fact_shipment,
    "fact_customer_sales": fact_customer_sales,
    "fact_product_sales": fact_product_sales
}

In [70]:
PROCESSED_PATH = f"s3://{BUCKET}/processed/combined_data/"

for name, df in dimensions.items():
    print(f"Writing {name}...")
    
    df.write \
        .mode("overwrite") \
        .parquet(f"{PROCESSED_PATH}{name}/")
    
    print(f"{name} done.")

Writing dim_customer...
dim_customer done.
Writing dim_product...
dim_product done.
Writing dim_category...
dim_category done.
Writing dim_department...
dim_department done.
Writing dim_supplier...
dim_supplier done.
Writing dim_employee...
dim_employee done.
Writing dim_shipper...
dim_shipper done.
Writing dim_date...
dim_date done.
Writing dim_payment_method...
dim_payment_method done.
Writing dim_order_status...
dim_order_status done.


In [71]:
for name, df in facts.items():
    print(f"Writing {name}...")
    
    df.write \
        .mode("overwrite") \
        .parquet(f"{PROCESSED_PATH}{name}/")
    
    print(f"{name} done.")

Writing fact_order...
fact_order done.
Writing fact_order_detail...
fact_order_detail done.
Writing fact_payment...
fact_payment done.
Writing fact_shipment...
fact_shipment done.
Writing fact_customer_sales...
fact_customer_sales done.
Writing fact_product_sales...
fact_product_sales done.
